# CYGNSS L1 fixed-operator 2020-2022 arm comparison

Extends `cygl1_fixedop_2020_7arm_omf_figures.ipynb` to the three-year bundle staged under
`output/cygl1_fixedop_2020_2022_stats` (see `README.md` and `cygl1_fixedop_2020_arm_comparison_report.md`
in that directory). Same Arizona domain (909 land tiles), same fixed-operator build.

Four arms were continued through 2022-12-31: the two coherency-filtered L1 arms (`full_xc015_coh040216`,
with and without errstd inflated to 3.9) and the two monitor-only arms (`l3`, `smap`). Three 2020-only
arms (the unfiltered benchmark, benchmark+err39, and the thinned `dense075_coh05` arm) are also in the
bundle for single-year context but are not the focus here.

**Headline finding from the report (§3): the spring degradation seen in 2020 is not a one-off — it
recurs every year, its calendar window moves, and it has two different mechanisms:**

- 2020 (May-Jun) and 2022 (Feb-May): **signal collapse** — corr(dF, inn) drops below ~0.1 and alpha_opt
  is near 0 (the increments carry essentially no information).
- 2021 (Apr-Jun): **overshoot** — corr stays normal (0.13-0.16) but the noise term is 3-5x the winter
  value and alpha_opt is 0.45-0.75 (the increments are real but roughly 1.5-2x too large).

This notebook is built to see both of those things directly: the moving spring window in the monthly
OmF-stdv and dMSE time series, and the corr/alpha_opt split that tells the two mechanisms apart.

Layout convention (unchanged from the 2020 notebook): systems are **L1, L3, ASCAT, SMOS, SMAP**; map
columns are matching OL, DA, `(DA - OL) / OL * 100`; negative = DA has lower OmF residuals (better).


In [ ]:
from __future__ import annotations

import pickle
from dataclasses import dataclass
from datetime import datetime
from pathlib import Path

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize, TwoSlopeNorm
from netCDF4 import Dataset
from IPython.display import display

REPO = Path.cwd()
if REPO.name != "geosldas-analysis":
    REPO = Path("/Users/amfox/Desktop/geosldas-analysis")
PROJECT = REPO / "projects" / "CYGNSS_L1_AZ"
STATS = PROJECT / "output" / "cygl1_fixedop_2020_2022_stats"
STATSDIR = STATS / "stats"
PERYEAR = STATSDIR / "per_year"
ONLY2020DIR = STATSDIR / "2020_only_arms"
SUMMARY = STATS / "summary"
OUT = STATS / "figures"
OUT.mkdir(parents=True, exist_ok=True)

MONTH_PERIOD = "202001_202212"      # 36 months, whole-period pkl
FILE_PERIOD = "20200101_20221231"   # whole-period nc4
PERIOD_LABEL = "2020-2022 (36 months)"
YEAR_FILE_PERIODS = {2020: "20200101_20201231", 2021: "20210101_20211231", 2022: "20220101_20221231"}

MONTH_PERIOD_2020 = "202001_202012"
FILE_PERIOD_2020 = "20200101_20201231"

NMIN = 10
MAP_EXTENT = (-118.6, -105.4, 28.6, 40.4)

plt.rcParams.update({
    "font.size": 10,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "figure.dpi": 120,
})


In [ ]:
@dataclass(frozen=True)
class Group:
    name: str
    indices: tuple[int, ...]
    units: str
    color: str


GROUPS = (
    Group("L1", (12,), "dB", "#7b3294"),
    Group("L3", (11,), "m$^3$ m$^{-3}$", "#d95f02"),
    Group("ASCAT", (8, 9, 10), "m$^3$ m$^{-3}$", "#1b9e77"),
    Group("SMOS", (0, 1, 2, 3), "K", "#7570b3"),
    Group("SMAP", (4, 5, 6, 7), "K", "#666666"),
)


@dataclass(frozen=True)
class Arm:
    tag: str
    exp_id: str
    label: str
    note: str


# Four arms run the full 2020-2022 period; these are the focus of this notebook.
CONTINUING_ARMS = (
    Arm("full_xc015_coh040216_fixedop", "DA_L1_full_xc015_coh040216_fixedop",
        "Filter only", "coherency_ratio filter 0.40-2.16, rebuilt clim, errstd 2.75 dB"),
    Arm("full_xc015_coh040216_err39_fixedop", "DA_L1_full_xc015_coh040216_err39_fixedop",
        "Filter + errstd 3.9 (best L1 arm)", "same filter, errstd 3.9"),
    Arm("l3_fixedop", "DA_L3_fixedop", "L3-only", "CYGNSS L3 SM only (L1 monitor-only)"),
    Arm("smap_fixedop", "DA_SMAP_fixedop", "SMAP-only", "SMAP L1C Tb only (L1 monitor-only)"),
)

# Three arms stopped at 2020-12-31 (same runs as the earlier 7-arm bundle); kept here only for
# single-year context in the headline table, not for the 3-year maps/time series below.
ARMS_2020_ONLY = (
    Arm("full_xc015_fixedop", "DA_L1_full_xc015_fixedop", "Benchmark (2020 only)",
        "full L1 stream, xcorr 0.15 deg, errstd 2.75 dB"),
    Arm("full_xc015_err39_fixedop", "DA_L1_full_xc015_err39_fixedop", "Benchmark, errstd 3.9 (2020 only)",
        "benchmark, errstd 3.9"),
    Arm("dense075_coh05_fixedop", "DA_L1_dense075_coh05_fixedop", "Thinned dense075/coh05 (2020 only)",
        "0.75 deg min separation, coherency >= 0.5, xcorr 0.625 deg"),
)

ARM_BY_TAG = {arm.tag: arm for arm in CONTINUING_ARMS + ARMS_2020_ONLY}
ARM_COLORS = {
    "full_xc015_coh040216_fixedop": "#c44e52",
    "full_xc015_coh040216_err39_fixedop": "#8172b2",
    "l3_fixedop": "#e17c05",
    "smap_fixedop": "#64b5cd",
}

BEST_TAG = "full_xc015_coh040216_err39_fixedop"
FILTER_TAG = "full_xc015_coh040216_fixedop"
L1_TAGS = (FILTER_TAG, BEST_TAG)


def continuing_arm_files(tag: str) -> dict[str, Path]:
    return {
        "ol_nc4": STATSDIR / f"temporal_stats_OL_paired_monitor_xmask_{tag}_{FILE_PERIOD}.nc4",
        "da_nc4": STATSDIR / f"temporal_stats_DA_paired_{tag}_{FILE_PERIOD}.nc4",
        "ol_pkl": STATSDIR / f"spatial_stats_OL_paired_monitor_xmask_{tag}_{MONTH_PERIOD}.pkl",
        "da_pkl": STATSDIR / f"spatial_stats_DA_paired_{tag}_{MONTH_PERIOD}.pkl",
    }


def year_files(tag: str, year: int) -> dict[str, Path]:
    suffix = YEAR_FILE_PERIODS[year]
    return {
        "ol_nc4": PERYEAR / f"temporal_stats_OL_paired_monitor_xmask_{tag}_{suffix}.nc4",
        "da_nc4": PERYEAR / f"temporal_stats_DA_paired_{tag}_{suffix}.nc4",
    }


def arm_2020_only_files(tag: str) -> dict[str, Path]:
    return {
        "ol_nc4": ONLY2020DIR / f"temporal_stats_OL_paired_monitor_xmask_{tag}_{FILE_PERIOD_2020}.nc4",
        "da_nc4": ONLY2020DIR / f"temporal_stats_DA_paired_{tag}_{FILE_PERIOD_2020}.nc4",
        "ol_pkl": ONLY2020DIR / f"spatial_stats_OL_paired_monitor_xmask_{tag}_{MONTH_PERIOD_2020}.pkl",
        "da_pkl": ONLY2020DIR / f"spatial_stats_DA_paired_{tag}_{MONTH_PERIOD_2020}.pkl",
    }


missing = []
for arm in CONTINUING_ARMS:
    for key, path in continuing_arm_files(arm.tag).items():
        if not path.exists():
            missing.append((arm.tag, key, path))
    for year in YEAR_FILE_PERIODS:
        for key, path in year_files(arm.tag, year).items():
            if not path.exists():
                missing.append((arm.tag, f"{key}_{year}", path))
for arm in ARMS_2020_ONLY:
    for key, path in arm_2020_only_files(arm.tag).items():
        if not path.exists():
            missing.append((arm.tag, key, path))
print(f"{len(CONTINUING_ARMS)} continuing arms, {len(ARMS_2020_ONLY)} 2020-only arms, {len(missing)} missing files")
for tag, key, path in missing:
    print("MISSING", tag, key, path)


In [ ]:
def read_nc(path: Path, names=("OmF_stdv", "OmF_mean", "O_mean", "F_mean", "N_data")) -> dict[str, np.ndarray]:
    with Dataset(path) as ds:
        return {
            name: np.ma.filled(ds.variables[name][:], np.nan).astype(float)
            for name in names
        }


def read_monthly(path: Path) -> dict[str, np.ndarray]:
    with path.open("rb") as stream:
        raw = pickle.load(stream)
    data = {
        key: (np.asarray(value, dtype=float) if key != "date_vec" else value)
        for key, value in raw.items()
    }
    data["months"] = np.asarray([datetime.strptime(str(v)[:6], "%Y%m") for v in data.get("date_vec", [])])
    return data


def weighted_species(values: np.ndarray, counts: np.ndarray, group: Group, nmin: int = NMIN) -> np.ndarray:
    """N-weighted average of a variable across the species that belong to one observing system."""
    idx = list(group.indices)
    group_vals = values[..., idx]
    group_counts = counts[..., idx]
    valid = np.isfinite(group_vals) & np.isfinite(group_counts) & (group_counts >= nmin)
    numerator = np.where(valid, group_vals * group_counts, 0.0).sum(axis=-1)
    denominator = np.where(valid, group_counts, 0.0).sum(axis=-1)
    return np.divide(numerator, denominator, out=np.full(denominator.shape, np.nan), where=denominator > 0)


def group_values(data: dict[str, np.ndarray], variable: str, group: Group) -> np.ndarray:
    return weighted_species(data[variable], data["N_data"], group)


def group_weight(data: dict[str, np.ndarray], group: Group) -> np.ndarray:
    return np.nansum(data["N_data"][..., list(group.indices)], axis=-1)


def percent_diff(da: np.ndarray, ol: np.ndarray) -> np.ndarray:
    return 100.0 * (da - ol) / ol


def n_weighted_mean(values: np.ndarray, weight: np.ndarray) -> float:
    valid = np.isfinite(values) & np.isfinite(weight) & (weight > 0)
    if not np.any(valid):
        return np.nan
    return float(np.average(values[valid], weights=weight[valid]))


def pooled_stdv(monthly: dict[str, np.ndarray], rows: slice, species: int) -> float:
    """N-weighted pooled OmF stdv across a slice of monthly rows, for one species (bundle README formula)."""
    N = monthly["N_data"][rows, species]
    m = monthly["OmF_mean"][rows, species]
    s = monthly["OmF_stdv"][rows, species]
    ok = (N > 0) & np.isfinite(m) & np.isfinite(s)
    N, m, s = N[ok], m[ok], s[ok]
    if N.sum() <= 0:
        return np.nan
    pooled_mean = (N * m).sum() / N.sum()
    return float(np.sqrt((N * (s**2 + m**2)).sum() / N.sum() - pooled_mean**2))


def pooled_group_pct(da_monthly, ol_monthly, rows: slice, group: Group) -> float:
    """Mean of the per-species pooled percent change, for the species in one system (README convention)."""
    pct = []
    for species in group.indices:
        da_s = pooled_stdv(da_monthly, rows, species)
        ol_s = pooled_stdv(ol_monthly, rows, species)
        if ol_s:
            pct.append(100.0 * (da_s - ol_s) / ol_s)
    return float(np.mean(pct)) if pct else np.nan


def norm_from_values(values, lower=2, upper=98):
    finite_parts = [np.ravel(v[np.isfinite(v)]) for v in values if np.any(np.isfinite(v))]
    if not finite_parts:
        return Normalize(0, 1)
    finite = np.concatenate(finite_parts)
    vmin, vmax = np.nanpercentile(finite, (lower, upper))
    if np.isclose(vmin, vmax):
        delta = max(abs(vmin) * 0.05, 1.0e-6)
        vmin -= delta
        vmax += delta
    return Normalize(vmin=vmin, vmax=vmax)


def signed_norm(values, limit=None, percentile=98):
    if limit is None:
        finite_parts = [np.ravel(np.abs(v[np.isfinite(v)])) for v in values if np.any(np.isfinite(v))]
        finite = np.concatenate(finite_parts) if finite_parts else np.asarray([1.0])
        limit = float(np.nanpercentile(finite, percentile))
    limit = max(float(limit), 1.0e-12)
    return TwoSlopeNorm(vmin=-limit, vcenter=0.0, vmax=limit)


def base_map(ax):
    ax.set_extent(MAP_EXTENT, crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND.with_scale("50m"), facecolor="0.94", zorder=0)
    ax.add_feature(cfeature.OCEAN.with_scale("50m"), facecolor="white", zorder=0)
    ax.add_feature(cfeature.COASTLINE.with_scale("50m"), lw=0.45, edgecolor="0.25")
    ax.add_feature(cfeature.BORDERS.with_scale("50m"), lw=0.35, edgecolor="0.35")
    states = cfeature.NaturalEarthFeature(
        "cultural", "admin_1_states_provinces_lakes", "50m", facecolor="none"
    )
    ax.add_feature(states, lw=0.35, edgecolor="0.45")


def scatter_tiles(ax, values, cmap, norm):
    valid = np.isfinite(values)
    return ax.scatter(
        lon[valid], lat[valid], c=values[valid], s=19, marker="s", linewidths=0,
        cmap=cmap, norm=norm, transform=ccrs.PlateCarree(), zorder=2,
    )


def setup_month_axis(ax, interval=4):
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=interval))
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
    ax.grid(True, lw=0.4, alpha=0.35)


# Spring degradation windows called out in the report (§3, finding 3): the calendar window moves
# every year, so we shade rather than assume a fixed month range.
SPRING_WINDOWS = {
    2020: ("2020-05-01", "2020-07-01", "signal collapse"),
    2021: ("2021-04-01", "2021-07-01", "overshoot"),
    2022: ("2022-02-01", "2022-06-01", "signal collapse"),
}


def shade_spring_windows(ax):
    for year, (start, end, _mechanism) in SPRING_WINDOWS.items():
        ax.axvspan(pd.Timestamp(start), pd.Timestamp(end), color="0.85", alpha=0.6, zorder=0)


In [ ]:
tile_coords = pd.read_csv(STATS / "tile_coords.csv").sort_values("tile_axis_index").reset_index(drop=True)
assert (tile_coords["tile_axis_index"].to_numpy() == np.arange(len(tile_coords))).all(), \
    "tile_coords.csv rows must already be in tile-axis order"
lon = tile_coords["com_lon"].to_numpy(dtype=float)
lat = tile_coords["com_lat"].to_numpy(dtype=float)

species_index = pd.read_csv(STATS / "species_index.csv")
for group in GROUPS:
    descrs = species_index.loc[list(group.indices), "descr"].tolist()
    print(f"{group.name:6s} idx={group.indices}  {descrs}")

print(f"\n{len(tile_coords)} tiles, lon [{lon.min():.2f}, {lon.max():.2f}], lat [{lat.min():.2f}, {lat.max():.2f}]")


In [ ]:
DATA = {}
for arm in CONTINUING_ARMS:
    files = continuing_arm_files(arm.tag)
    DATA[arm.tag] = {
        "temporal": {"ol": read_nc(files["ol_nc4"]), "da": read_nc(files["da_nc4"])},
        "monthly": {"ol": read_monthly(files["ol_pkl"]), "da": read_monthly(files["da_pkl"])},
        "temporal_year": {
            year: {"ol": read_nc(year_files(arm.tag, year)["ol_nc4"]), "da": read_nc(year_files(arm.tag, year)["da_nc4"])}
            for year in YEAR_FILE_PERIODS
        },
    }

for arm in CONTINUING_ARMS:
    n_tile = DATA[arm.tag]["temporal"]["da"]["OmF_stdv"].shape[0]
    n_month = DATA[arm.tag]["monthly"]["da"]["OmF_stdv"].shape[0]
    print(f"{arm.tag:38s} tiles={n_tile:4d}  months={n_month:2d}  ({arm.label})")

DATA_2020_ONLY = {}
for arm in ARMS_2020_ONLY:
    files = arm_2020_only_files(arm.tag)
    DATA_2020_ONLY[arm.tag] = {
        "temporal": {"ol": read_nc(files["ol_nc4"]), "da": read_nc(files["da_nc4"])},
        "monthly": {"ol": read_monthly(files["ol_pkl"]), "da": read_monthly(files["da_pkl"])},
    }
print("loaded", len(DATA_2020_ONLY), "2020-only arms for headline context")


## Headline: does the 2020 impact hold up over 2021-2022?

Computed directly from the whole-period (36-month) `spatial_stats_*.pkl` files using the bundle
README's own pooling formula (N-weighted pooled stdv per species, then the mean of the per-species
percent changes within a system) — the same method that produced the report's headline table, so
these numbers should match it exactly.


In [ ]:
YEAR_ROWS = {2020: slice(0, 12), 2021: slice(12, 24), 2022: slice(24, 36), "2020-22": slice(0, 36)}

headline_rows = []
for arm in CONTINUING_ARMS:
    da_m, ol_m = DATA[arm.tag]["monthly"]["da"], DATA[arm.tag]["monthly"]["ol"]
    for period, rows in YEAR_ROWS.items():
        row = {"arm": arm.tag, "period": period}
        for group in GROUPS:
            row[group.name] = pooled_group_pct(da_m, ol_m, rows, group)
        headline_rows.append(row)

headline = pd.DataFrame(headline_rows)
group_cols = [g.name for g in GROUPS]
display(headline.style.format({c: "{:+.2f}" for c in group_cols}))

# sanity check against the bundle README's worked example (SMAP, err39 arm, 2021 == -0.70%)
check = headline.query("arm == @BEST_TAG and period == 2021")["SMAP"].iloc[0]
print(f"\nsanity check: SMAP 2021 for {BEST_TAG} = {check:+.2f}% (README worked example: -0.70%)")


In [ ]:
pooled_2022 = headline[headline["period"] == "2020-22"].set_index("arm").loc[[a.tag for a in CONTINUING_ARMS], group_cols]
pooled_2022.index = [ARM_BY_TAG[tag].label for tag in pooled_2022.index]

fig, (ax_heat, ax_trend) = plt.subplots(1, 2, figsize=(13.5, 4.4), gridspec_kw={"width_ratios": [1, 1.5]})

matrix = pooled_2022.to_numpy()
heat_norm = signed_norm([matrix])
image = ax_heat.imshow(matrix, cmap="RdBu_r", norm=heat_norm, aspect="auto")
ax_heat.set_xticks(range(len(group_cols)))
ax_heat.set_xticklabels(group_cols)
ax_heat.set_yticks(range(len(pooled_2022)))
ax_heat.set_yticklabels(pooled_2022.index, fontsize=8.5)
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        ax_heat.text(j, i, f"{matrix[i, j]:+.2f}", ha="center", va="center", fontsize=8)
ax_heat.set_title("Pooled 2020-2022 OmF StDev % change vs OL", fontweight="bold")
fig.colorbar(image, ax=ax_heat, orientation="vertical", fraction=0.05, pad=0.02, label="%")

for group in GROUPS:
    for arm in CONTINUING_ARMS:
        yearly = headline[(headline["arm"] == arm.tag) & (headline["period"].isin([2020, 2021, 2022]))]
        yearly = yearly.set_index("period").loc[[2020, 2021, 2022], group.name]
        if group.name == "SMAP":  # only annotate one group's lines to keep the legend readable
            ax_trend.plot([2020, 2021, 2022], yearly, "-o", color=ARM_COLORS[arm.tag], lw=1.8, ms=4, label=arm.label)
ax_trend.axhline(0, color="0.25", lw=0.8)
ax_trend.set_xticks([2020, 2021, 2022])
ax_trend.set_ylabel("SMAP OmF StDev % change vs OL")
ax_trend.set_title("Year-by-year trend (SMAP shown; same pattern for SMOS/L3)", fontweight="bold")
ax_trend.legend(frameon=False, fontsize=8)

fig.suptitle(f"Headline summary, {PERIOD_LABEL}", y=1.03, fontsize=13)
fig.tight_layout()
headline_path = OUT / "headline_omf_stdv_pct_heatmap_and_trend.png"
fig.savefig(headline_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", headline_path.relative_to(PROJECT))


For single-year context, here are the three arms that stopped at 2020-12-31 (same runs covered in the
earlier 7-arm notebook), computed the same way from their 12-month pkl files.


In [ ]:
rows_2020_only = []
for arm in ARMS_2020_ONLY:
    da_m, ol_m = DATA_2020_ONLY[arm.tag]["monthly"]["da"], DATA_2020_ONLY[arm.tag]["monthly"]["ol"]
    row = {"arm": arm.tag, "period": 2020}
    for group in GROUPS:
        row[group.name] = pooled_group_pct(da_m, ol_m, slice(0, 12), group)
    rows_2020_only.append(row)

headline_2020_only = pd.DataFrame(rows_2020_only)
display(headline_2020_only.style.format({c: "{:+.2f}" for c in group_cols}))


## Does the domain-wide comparison include tiles L1 never observes?

Every percent-change number above is a weighted mean over each *species'* own valid-tile footprint —
but the sensors do not share the same footprint within this 909-tile AZ box. In particular, CYGNSS L1
is not global-coverage: its orbital inclination limits it to roughly ±38 degrees latitude, and this
domain runs from 29 to 40 degrees North. So before trusting a "domain-wide" SMOS/SMAP number for an L1
arm, it's worth checking how much of the domain L1 actually reaches — and whether the comparison against
L3 (which has a different, only partially-overlapping footprint) is being run over tiles where L1 could
plausibly have had any effect at all.

(This is the same idea as the `LAT_BOUNDARY = 37.5` restriction in the original
`final_dense075_coh05_omf_figures.ipynb` notebook, which annotated every map with the area-weighted mean
*south of 37.5N* rather than the full-domain mean. That notebook was built for a single L1 arm, so a
fixed latitude cutoff was a reasonable proxy. Here we derive the footprint directly from each arm's own
`N_data`, which is more precise and doesn't assume the cutoff is purely a latitude line.)


In [ ]:
best_da_temporal = DATA[BEST_TAG]["temporal"]["da"]
footprint_rows = []
for i, row in species_index.iterrows():
    n = best_da_temporal["N_data"][:, i]
    group_name = next((g.name for g in GROUPS if i in g.indices), None)
    footprint_rows.append({"species": row["descr"], "group": group_name, "tiles_with_obs": int(np.sum(n > 0))})
species_footprint = pd.DataFrame(footprint_rows)
display(species_footprint)

group_footprint_rows = []
for group in GROUPS:
    any_obs = np.any(best_da_temporal["N_data"][:, list(group.indices)] > 0, axis=-1)
    group_footprint_rows.append({
        "group": group.name, "tiles_with_obs": int(any_obs.sum()),
        "pct_of_domain": 100 * any_obs.sum() / len(any_obs),
    })
print(f"\n{len(tile_coords)}-tile domain, coverage by system ({ARM_BY_TAG[BEST_TAG].label}):")
display(pd.DataFrame(group_footprint_rows).style.format({"pct_of_domain": "{:.0f}%"}))

# CYGNSS L1's own footprint: this is the mask we'll use to restrict the L1-vs-L3 comparison below.
L1_FOOTPRINT = best_da_temporal["N_data"][:, 12] > 0
print(f"\nL1 footprint: {L1_FOOTPRINT.sum()} of {len(L1_FOOTPRINT)} tiles ({100 * L1_FOOTPRINT.mean():.0f}%)")

lat_bins = np.arange(29, 41, 1)
print("\nL1 coverage by 1-degree latitude band:")
for b0, b1 in zip(lat_bins[:-1], lat_bins[1:]):
    band = (lat >= b0) & (lat < b1)
    n_band, n_l1 = band.sum(), (band & L1_FOOTPRINT).sum()
    print(f"  [{b0:.0f}, {b1:.0f}) N: {n_band:3d} tiles, {n_l1:3d} with L1 obs ({100 * n_l1 / max(n_band, 1):.0f}%)")


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.8), subplot_kw={"projection": ccrs.PlateCarree()})
base_map(ax)
ax.scatter(lon[~L1_FOOTPRINT], lat[~L1_FOOTPRINT], c="0.75", s=19, marker="s", linewidths=0,
           transform=ccrs.PlateCarree(), zorder=2, label=f"no L1 obs ({int((~L1_FOOTPRINT).sum())} tiles)")
ax.scatter(lon[L1_FOOTPRINT], lat[L1_FOOTPRINT], c="#7b3294", s=19, marker="s", linewidths=0,
           transform=ccrs.PlateCarree(), zorder=2, label=f"has L1 obs ({int(L1_FOOTPRINT.sum())} tiles)")
ax.legend(loc="lower left", fontsize=8, framealpha=0.9)
ax.set_title(
    f"CYGNSS L1 observational footprint, {PERIOD_LABEL}\n"
    f"{L1_FOOTPRINT.sum()} of {len(L1_FOOTPRINT)} tiles ({100 * L1_FOOTPRINT.mean():.0f}%) "
    "-- note the cutoff above ~37.4N (orbital inclination)",
    fontweight="bold", fontsize=10,
)
footprint_path = OUT / "l1_observational_footprint_map.png"
fig.savefig(footprint_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", footprint_path.relative_to(PROJECT))


### L1 vs L3, restricted to the region L1 actually observes

The two prior sections computed each system's domain mean over its own full native footprint. Here we
repeat that for the best L1 arm and the L3-only arm, but restrict the SMOS/SMAP/ASCAT/L3 domain means to
only the 563 tiles where CYGNSS L1 has an observation — the same tile mask for both arms, so this is a
fair, apples-to-apples test of "in the region L1 could plausibly influence, which observation type gives
the bigger benefit." (`n_weighted_mean` already accepts a weight array, so restricting to the footprint
is just an elementwise multiply by the boolean mask — the same per-tile N-weighting is used either way.)


In [ ]:
footprint_comparison_rows = []
for arm_tag in (BEST_TAG, "l3_fixedop"):
    temporal = DATA[arm_tag]["temporal"]
    for group in GROUPS:
        ol = group_values(temporal["ol"], "OmF_stdv", group)
        da = group_values(temporal["da"], "OmF_stdv", group)
        pct = percent_diff(da, ol)
        weight = group_weight(temporal["da"], group)
        footprint_comparison_rows.append({
            "arm": ARM_BY_TAG[arm_tag].label,
            "system": group.name,
            "full_domain_pct": n_weighted_mean(pct, weight),
            "n_tiles_full_domain": int(np.sum(np.isfinite(pct) & (weight > 0))),
            "l1_footprint_only_pct": n_weighted_mean(pct, weight * L1_FOOTPRINT),
            "n_tiles_l1_footprint": int(np.sum(np.isfinite(pct) & (weight > 0) & L1_FOOTPRINT)),
        })

footprint_comparison = pd.DataFrame(footprint_comparison_rows)
display(footprint_comparison.style.format({"full_domain_pct": "{:+.2f}", "l1_footprint_only_pct": "{:+.2f}"}))

footprint_comparison_path = OUT / "l1_vs_l3_footprint_restricted_comparison.csv"
footprint_comparison.to_csv(footprint_comparison_path, index=False)
print("wrote", footprint_comparison_path.relative_to(PROJECT))


**Restricting to the L1 footprint does not change the conclusion — if anything it sharpens it.** Every
system moves toward a slightly *larger*-magnitude effect once the un-reachable northern tiles are
dropped (consistent with that region being generally less dynamic for every sensor, not specifically an
L1 blind spot diluting a hidden win): L3-only's ASCAT improvement gets a little stronger (not weaker),
and the best L1 arm's ASCAT penalty gets a little worse, not better. L3-only still wins on SMOS, SMAP and
ASCAT restricted to exactly the tiles L1 can see, and the two arms remain close only on L1's own species.
So the coverage gap was a legitimate thing to check, but it was not masking a hidden L1 advantage.


## The moving spring window: 36 months, all four continuing arms

One panel per observing system, one line per arm, January 2020 through December 2022. Shaded bands
mark the spring windows the report identifies as degraded for the L1 arms: May-Jun 2020, Apr-Jun 2021,
Feb-Jun 2022 (the window moves — a fixed calendar gate would not catch all three).


In [ ]:
fig, axes = plt.subplots(len(GROUPS), 1, figsize=(12.5, 13.5), sharex=True)
for ax, group in zip(axes, GROUPS):
    shade_spring_windows(ax)
    ax.axhline(0, color="0.25", lw=0.8, zorder=1)
    for arm in CONTINUING_ARMS:
        monthly = DATA[arm.tag]["monthly"]
        months = monthly["ol"]["months"]
        ol = group_values(monthly["ol"], "OmF_stdv", group)
        da = group_values(monthly["da"], "OmF_stdv", group)
        pct = percent_diff(da, ol)
        ax.plot(months, pct, "-o", color=ARM_COLORS[arm.tag], lw=1.6, ms=3.0, label=arm.label, zorder=2)
    ax.set_ylabel(f"{group.name}\n%", fontweight="bold")
    setup_month_axis(ax)

axes[-1].set_xlabel("month")
for tick in axes[-1].get_xticklabels():
    tick.set_rotation(35)
    tick.set_ha("right")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.995), fontsize=8.5)
fig.suptitle(
    f"Monthly OmF StDev change vs matching OL, {PERIOD_LABEL}\n"
    "100 x (DA - OL) / OL; negative = lower DA residual; shaded = reported spring degradation window",
    y=1.035, fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
monthly_pct_path = OUT / "cross_arm_monthly_omf_stdv_percent_36mo_5x1.png"
fig.savefig(monthly_pct_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", monthly_pct_path.relative_to(PROJECT))


## Per-tile maps: pooled 2020-2022 percent-difference, all four continuing arms

From the whole-period `temporal_stats_*.nc4` files (all 909 tiles, pooled over the full three years).


In [ ]:
combined_rows = []
combined_summary_rows = []
for arm in CONTINUING_ARMS:
    temporal = DATA[arm.tag]["temporal"]
    row = []
    for group in GROUPS:
        ol = group_values(temporal["ol"], "OmF_stdv", group)
        da = group_values(temporal["da"], "OmF_stdv", group)
        pct = percent_diff(da, ol)
        weight = group_weight(temporal["da"], group)
        mean_value = n_weighted_mean(pct, weight)
        row.append((group, pct, mean_value))
        combined_summary_rows.append({"arm": arm.tag, "system": group.name, "n_weighted_percent_da_minus_ol": mean_value})
    combined_rows.append((arm, row))

combined_summary = pd.DataFrame(combined_summary_rows)
combined_summary_path = OUT / "cross_arm_omf_stdv_percent_maps_4x5_summary.csv"
combined_summary.to_csv(combined_summary_path, index=False)

fig, axes = plt.subplots(
    len(CONTINUING_ARMS), len(GROUPS), figsize=(15.0, 11.5),
    subplot_kw={"projection": ccrs.PlateCarree()},
)
combined_norm = signed_norm([pct for _, row in combined_rows for _, pct, _ in row])
for row_index, (arm, row) in enumerate(combined_rows):
    for col_index, (group, pct, mean_value) in enumerate(row):
        ax = axes[row_index, col_index]
        base_map(ax)
        combined_image = scatter_tiles(ax, pct, "RdBu_r", combined_norm)
        if row_index == 0:
            ax.set_title(group.name, fontweight="bold", pad=6)
        if col_index == 0:
            ax.text(-0.16, 0.5, arm.label, transform=ax.transAxes, rotation=90,
                    va="center", ha="center", fontweight="bold", fontsize=9)
        ax.text(0.02, 0.03, f"{mean_value:+.2f}%", transform=ax.transAxes, fontsize=8,
                bbox={"facecolor": "white", "edgecolor": "0.7", "alpha": 0.9, "pad": 1.5})
        ax.set_xticks([])
        ax.set_yticks([])

colorbar_ax = fig.add_axes((0.2, 0.03, 0.62, 0.016))
colorbar = fig.colorbar(combined_image, cax=colorbar_ax, orientation="horizontal")
colorbar.set_label("(DA - OL) / OL OmF StDev (%), pooled 2020-2022; negative = lower DA residual", fontsize=9)
colorbar.ax.tick_params(labelsize=8)
fig.suptitle(f"OmF standard deviation change vs matching OL, {PERIOD_LABEL}", y=0.99, fontsize=14)
fig.subplots_adjust(top=0.94, bottom=0.09, left=0.1, right=0.99, hspace=0.15, wspace=0.08)
combined_map_path = OUT / "cross_arm_omf_stdv_percent_maps_4x5.png"
fig.savefig(combined_map_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", combined_map_path.relative_to(PROJECT))
print("wrote", combined_summary_path.relative_to(PROJECT))


## Does the spring problem move geographically too? Best L1 arm, year by year

Annual (not spring-only — the per-tile files are only available at annual resolution) percent-difference
maps for the best L1 arm, one row per calendar year, from `stats/per_year/`. This shows whether the
*location* of the DA's impact shifts as the spring window moves, even though it can't isolate the spring
months spatially on their own.


In [ ]:
year_rows = []
year_summary_rows = []
for year in (2020, 2021, 2022):
    temporal = DATA[BEST_TAG]["temporal_year"][year]
    row = []
    for group in GROUPS:
        ol = group_values(temporal["ol"], "OmF_stdv", group)
        da = group_values(temporal["da"], "OmF_stdv", group)
        pct = percent_diff(da, ol)
        weight = group_weight(temporal["da"], group)
        mean_value = n_weighted_mean(pct, weight)
        row.append((group, pct, mean_value))
        year_summary_rows.append({"year": year, "system": group.name, "n_weighted_percent_da_minus_ol": mean_value})
    year_rows.append((year, row))

year_summary = pd.DataFrame(year_summary_rows)
year_summary_path = OUT / f"{BEST_TAG}_per_year_omf_stdv_percent_maps_summary.csv"
year_summary.to_csv(year_summary_path, index=False)

fig, axes = plt.subplots(3, len(GROUPS), figsize=(15.0, 9.0), subplot_kw={"projection": ccrs.PlateCarree()})
year_norm = signed_norm([pct for _, row in year_rows for _, pct, _ in row])
for row_index, (year, row) in enumerate(year_rows):
    for col_index, (group, pct, mean_value) in enumerate(row):
        ax = axes[row_index, col_index]
        base_map(ax)
        year_image = scatter_tiles(ax, pct, "RdBu_r", year_norm)
        if row_index == 0:
            ax.set_title(group.name, fontweight="bold", pad=6)
        if col_index == 0:
            ax.text(-0.16, 0.5, str(year), transform=ax.transAxes, rotation=90,
                    va="center", ha="center", fontweight="bold", fontsize=10)
        ax.text(0.02, 0.03, f"{mean_value:+.2f}%", transform=ax.transAxes, fontsize=8,
                bbox={"facecolor": "white", "edgecolor": "0.7", "alpha": 0.9, "pad": 1.5})
        ax.set_xticks([])
        ax.set_yticks([])

colorbar_ax = fig.add_axes((0.2, 0.02, 0.62, 0.022))
colorbar = fig.colorbar(year_image, cax=colorbar_ax, orientation="horizontal")
colorbar.set_label(f"(DA - OL) / OL OmF StDev (%), annual; {ARM_BY_TAG[BEST_TAG].label}", fontsize=9)
colorbar.ax.tick_params(labelsize=8)
fig.suptitle(f"{ARM_BY_TAG[BEST_TAG].label}: annual OmF StDev change by year", y=0.995, fontsize=13)
fig.subplots_adjust(top=0.93, bottom=0.1, left=0.1, right=0.99, hspace=0.15, wspace=0.08)
year_map_path = OUT / f"{BEST_TAG}_per_year_omf_stdv_percent_maps_3x5.png"
fig.savefig(year_map_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", year_map_path.relative_to(PROJECT))
print("wrote", year_summary_path.relative_to(PROJECT))


## Per-tile monthly time series, best L1 arm, all 36 months

Domain-wide monthly OmF StDev (absolute values, N-weighted across species in each system) for OL vs DA,
plus the percent difference, from the whole-period `spatial_stats_*.pkl`.


In [ ]:
def plot_arm_monthly_omf_stdv(tag: str):
    arm = ARM_BY_TAG[tag]
    monthly = DATA[tag]["monthly"]
    months = monthly["ol"]["months"]
    assert np.array_equal(months, monthly["da"]["months"]), f"month vectors differ for {tag}"

    fig, axes = plt.subplots(5, 2, figsize=(13.5, 12.8), sharex=True)
    summary_rows = []
    for row_index, group in enumerate(GROUPS):
        left, right = axes[row_index, 0], axes[row_index, 1]
        shade_spring_windows(left)
        shade_spring_windows(right)
        ol = group_values(monthly["ol"], "OmF_stdv", group)
        da = group_values(monthly["da"], "OmF_stdv", group)
        pct = percent_diff(da, ol)

        left.plot(months, ol, "--o", color="0.35", lw=1.3, ms=2.8, label="matching OL" if row_index == 0 else None, zorder=2)
        left.plot(months, da, "-o", color=group.color, lw=1.7, ms=3.2, label="DA" if row_index == 0 else None, zorder=2)
        right.plot(months, pct, "-o", color=group.color, lw=1.7, ms=3.2, zorder=2)
        right.axhline(0, color="0.25", lw=0.8, zorder=1)

        left.set_ylabel(f"{group.name}\n{group.units}", fontweight="bold")
        right.set_ylabel("%")
        setup_month_axis(left)
        setup_month_axis(right)
        if row_index == 0:
            left.set_title("OmF StDev: matching OL and DA", fontweight="bold")
            right.set_title("Percent difference from matching OL", fontweight="bold")
        if row_index == len(GROUPS) - 1:
            left.set_xlabel("month")
            right.set_xlabel("month")
        for axis in (left, right):
            for label in axis.get_xticklabels():
                label.set_rotation(35)
                label.set_ha("right")

        valid = np.isfinite(pct)
        summary_rows.append({
            "arm": tag, "system": group.name,
            "monthly_mean_percent_da_minus_ol": float(np.nanmean(pct)),
            "months_improved_da_lt_ol": int(np.sum(pct[valid] < 0)),
            "months_valid": int(valid.sum()),
        })

    handles, labels = axes[0, 0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=2, frameon=False, bbox_to_anchor=(0.5, 0.985))
    fig.suptitle(
        f"{arm.label} ({tag}): monthly OmF StDev, {PERIOD_LABEL}\n"
        "Percent difference uses 100 x (DA - OL) / OL; shaded = reported spring degradation window",
        y=1.025, fontsize=14,
    )
    fig.tight_layout(rect=(0, 0, 1, 0.95))
    path = OUT / f"{tag}_monthly_omf_stdv_36mo_5x2.png"
    fig.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()

    table = pd.DataFrame(summary_rows)
    table_path = OUT / f"{tag}_monthly_omf_stdv_36mo_summary.csv"
    table.to_csv(table_path, index=False)
    print("wrote", path.relative_to(PROJECT))
    print("wrote", table_path.relative_to(PROJECT))
    return table


display(plot_arm_monthly_omf_stdv(BEST_TAG))


## Noise vs. gain: telling the two spring mechanisms apart

From `summary/noise_gain_<tag>_202001_202212.csv` (same decomposition as the 2020 notebook, now over
36 months): `inn = O - F_OL`, `dF = F_DA - F_OL`, `dMSE = noise + gain` as % of the OL MSE,
`corr = corr(dF, inn)`, `alpha_opt` = increment scaling that would minimize MSE (near 0 = no
information; well above 1 = increments too small; well below 1 but well above 0 with normal noise =
increments too large).

If a spring window shows **corr collapsing toward 0**, that's the 2020/2022 "signal collapse" pattern
— the L1-driven increments simply aren't correlated with the real error anymore. If **corr stays normal
but noise spikes and alpha_opt drops toward 0 from a large increment**, that's the 2021 "overshoot"
pattern — the correction is real but too big.


In [ ]:
NOISE_GAIN = {}
for arm in CONTINUING_ARMS:
    path = SUMMARY / f"noise_gain_{arm.tag}_{MONTH_PERIOD}.csv"
    df = pd.read_csv(path)
    df["month_dt"] = pd.to_datetime(df["month"], format="%Y-%m", errors="coerce")
    NOISE_GAIN[arm.tag] = df

fig, axes = plt.subplots(3, 1, figsize=(12.5, 9.5), sharex=True)
for arm_tag, color, label in [(FILTER_TAG, ARM_COLORS[FILTER_TAG], ARM_BY_TAG[FILTER_TAG].label),
                                (BEST_TAG, ARM_COLORS[BEST_TAG], ARM_BY_TAG[BEST_TAG].label)]:
    rows = NOISE_GAIN[arm_tag]
    smap_rows = rows[(rows["grp"] == "SMAP") & rows["month_dt"].notna()].sort_values("month_dt")
    axes[0].plot(smap_rows["month_dt"], smap_rows["corr"], "-o", color=color, lw=1.7, ms=3.2, label=label)
    axes[1].plot(smap_rows["month_dt"], smap_rows["alpha_opt"], "-o", color=color, lw=1.7, ms=3.2, label=label)
    axes[2].plot(smap_rows["month_dt"], smap_rows["noise"], "-o", color=color, lw=1.7, ms=3.2, label=label)

for ax in axes:
    shade_spring_windows(ax)
axes[0].axhline(0.1, color="0.4", lw=0.8, ls=":")
axes[0].set_ylabel("corr(dF, inn)\nSMAP", fontweight="bold")
axes[1].axhline(1.0, color="0.4", lw=0.8, ls=":")
axes[1].axhline(0.0, color="0.25", lw=0.8)
axes[1].set_ylabel("alpha_opt\nSMAP", fontweight="bold")
axes[2].set_ylabel("noise\n(% of OL MSE)\nSMAP", fontweight="bold")
axes[2].set_xlabel("month")
for ax in axes:
    setup_month_axis(ax)
for tick in axes[-1].get_xticklabels():
    tick.set_rotation(35)
    tick.set_ha("right")
axes[0].legend(frameon=False, fontsize=8.5, loc="lower left")
fig.suptitle(
    "SMAP corr(dF, inn), alpha_opt and noise, both L1 arms, 2020-2022\n"
    "shaded = reported spring degradation window (dotted lines: corr=0.1, alpha_opt=1 reference)",
    y=1.02, fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.95))
mechanism_path = OUT / "l1_arms_smap_corr_alpha_noise_36mo.png"
fig.savefig(mechanism_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", mechanism_path.relative_to(PROJECT))


In [ ]:
mechanism_rows = []
for arm_tag in L1_TAGS:
    rows = NOISE_GAIN[arm_tag]
    rows = rows[(rows["grp"] == "SMAP") & rows["month_dt"].notna()]
    for year, (start, end, mechanism) in SPRING_WINDOWS.items():
        window = rows[(rows["month_dt"] >= start) & (rows["month_dt"] < end)]
        mechanism_rows.append({
            "arm": ARM_BY_TAG[arm_tag].label,
            "window": f"{year} ({mechanism})",
            "corr": window["corr"].mean(),
            "alpha_opt": window["alpha_opt"].mean(),
            "noise_pct_of_OL_MSE": window["noise"].mean(),
        })
mechanism_table = pd.DataFrame(mechanism_rows)

fig, (ax_corr, ax_alpha) = plt.subplots(1, 2, figsize=(11.0, 3.6))
pivot_corr = mechanism_table.pivot(index="arm", columns="window", values="corr")
pivot_alpha = mechanism_table.pivot(index="arm", columns="window", values="alpha_opt")
for ax, pivot, title, cmap, vmin, vmax in (
    (ax_corr, pivot_corr, "corr(dF, inn), SMAP", "viridis", 0, max(0.3, pivot_corr.to_numpy().max())),
    (ax_alpha, pivot_alpha, "alpha_opt, SMAP", "RdBu_r", -0.5, 1.0),
):
    image = ax.imshow(pivot.to_numpy(), cmap=cmap, vmin=vmin, vmax=vmax, aspect="auto")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns, fontsize=8, rotation=20, ha="right")
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels(pivot.index, fontsize=8)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.iat[i, j]:.2f}", ha="center", va="center", fontsize=8.5)
    ax.set_title(title, fontweight="bold", fontsize=10)
    fig.colorbar(image, ax=ax, fraction=0.06, pad=0.03)

fig.suptitle("Spring window means by mechanism (report §3, finding 8)", y=1.05, fontsize=12)
fig.tight_layout()
mechanism_summary_path = OUT / "l1_arms_spring_mechanism_summary.csv"
mechanism_table.to_csv(mechanism_summary_path, index=False)
mechanism_fig_path = OUT / "l1_arms_spring_mechanism_heatmap.png"
fig.savefig(mechanism_fig_path, dpi=180, bbox_inches="tight")
plt.show()
display(mechanism_table)
print("wrote", mechanism_fig_path.relative_to(PROJECT))
print("wrote", mechanism_summary_path.relative_to(PROJECT))


## Cross-arm monthly dMSE and pooled 36-month corr, all four continuing arms


In [ ]:
fig, axes = plt.subplots(len(GROUPS), 1, figsize=(12.5, 13.5), sharex=True)
for ax, group in zip(axes, GROUPS):
    shade_spring_windows(ax)
    ax.axhline(0, color="0.25", lw=0.8, zorder=1)
    for arm in CONTINUING_ARMS:
        rows = NOISE_GAIN[arm.tag]
        monthly = rows[(rows["grp"] == group.name) & rows["month_dt"].notna()].sort_values("month_dt")
        ax.plot(monthly["month_dt"], monthly["dMSE"], "-o", color=ARM_COLORS[arm.tag], lw=1.5, ms=2.8, label=arm.label, zorder=2)
    ax.set_ylabel(f"{group.name}\ndMSE %", fontweight="bold")
    setup_month_axis(ax)

axes[-1].set_xlabel("month")
for tick in axes[-1].get_xticklabels():
    tick.set_rotation(35)
    tick.set_ha("right")
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.995), fontsize=8.5)
fig.suptitle(
    f"Monthly MSE change vs matching OL (noise + gain decomposition), {PERIOD_LABEL}\n"
    "dMSE as % of the OL MSE; negative = DA reduces MSE",
    y=1.035, fontsize=13,
)
fig.tight_layout(rect=(0, 0, 1, 0.96))
dmse_path = OUT / "cross_arm_monthly_dmse_percent_36mo_5x1.png"
fig.savefig(dmse_path, dpi=180, bbox_inches="tight")
plt.show()
print("wrote", dmse_path.relative_to(PROJECT))


In [ ]:
full_period_noise_gain = pd.concat(
    [NOISE_GAIN[arm.tag][NOISE_GAIN[arm.tag]["month"] == "all"].assign(arm=arm.tag) for arm in CONTINUING_ARMS],
    ignore_index=True,
)
full_period_pivot = full_period_noise_gain.pivot(index="arm", columns="grp", values="corr").loc[
    [arm.tag for arm in CONTINUING_ARMS], [g.name for g in GROUPS]
]
full_period_pivot.index = [ARM_BY_TAG[tag].label for tag in full_period_pivot.index]

fig, ax = plt.subplots(figsize=(7.0, 3.6))
image = ax.imshow(full_period_pivot.to_numpy(), cmap="viridis", vmin=0, vmax=full_period_pivot.to_numpy().max())
ax.set_xticks(range(len(GROUPS)))
ax.set_xticklabels([g.name for g in GROUPS])
ax.set_yticks(range(len(full_period_pivot)))
ax.set_yticklabels(full_period_pivot.index, fontsize=8.5)
for i in range(full_period_pivot.shape[0]):
    for j in range(full_period_pivot.shape[1]):
        val = full_period_pivot.iat[i, j]
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=8,
                 color="white" if val < full_period_pivot.to_numpy().max() * 0.6 else "black")
ax.set_title(f"Pooled 36-month corr(dF, inn), {PERIOD_LABEL}", fontweight="bold", fontsize=10)
fig.colorbar(image, ax=ax, fraction=0.05, pad=0.02, label="corr(dF, inn)")
fig.tight_layout()
corr_path = OUT / "cross_arm_36mo_corr_heatmap.png"
fig.savefig(corr_path, dpi=180, bbox_inches="tight")
plt.show()

noise_gain_summary_path = OUT / "cross_arm_36mo_noise_gain_summary.csv"
full_period_noise_gain.to_csv(noise_gain_summary_path, index=False)
display(full_period_noise_gain[["arm", "grp", "N", "dMSE", "noise", "gain", "corr", "alpha_opt"]])
print("wrote", corr_path.relative_to(PROJECT))
print("wrote", noise_gain_summary_path.relative_to(PROJECT))


Saved outputs (all under `output/cygl1_fixedop_2020_2022_stats/figures/`):

- `headline_omf_stdv_pct_heatmap_and_trend.png`
- `cross_arm_monthly_omf_stdv_percent_36mo_5x1.png` (the moving spring window, all 4 arms)
- `cross_arm_omf_stdv_percent_maps_4x5.png` (+ `*_summary.csv`) — pooled 2020-2022 maps
- `<best_tag>_per_year_omf_stdv_percent_maps_3x5.png` (+ `*_summary.csv`) — year-by-year maps
- `<best_tag>_monthly_omf_stdv_36mo_5x2.png` (+ `*_summary.csv`)
- `l1_arms_smap_corr_alpha_noise_36mo.png` — the two-mechanism diagnostic
- `l1_arms_spring_mechanism_heatmap.png` (+ `*_summary.csv`) — reproduces report §3 finding 8's table
- `cross_arm_monthly_dmse_percent_36mo_5x1.png`
- `cross_arm_36mo_corr_heatmap.png` (+ `*_summary.csv`)

For the full write-up, see `cygl1_fixedop_2020_arm_comparison_report.md` §3-4 in the same directory
(the recommended next step there is a seasonal/adaptive L1 R-weighting experiment, since a fixed
calendar gate would miss 2022's Feb-May window).
